安装库和配置 API Key

In [ ]:
import torch
import torch.nn as nn
from torchvision import models as tv_models
import os

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Download Places365 Model Weights and Category File

We need to download the pre-trained ResNet50 model weights for Places365 and its corresponding category list. These files will be saved in a new directory `/content/places365`.

In [ ]:
weights_url = "http://places2.csail.mit.edu/models_places365/resnet50_places365.pth.tar"
category_url = "https://raw.githubusercontent.com/CSAILVision/places365challenge/master/categories_places365.txt"

# Define local paths
places365_dir = "/content/places365"
weight_path = os.path.join(places365_dir, "resnet50_places365.pth.tar")
category_path = os.path.join(places365_dir, "categories_places365.txt")

# Create directory if it doesn't exist
os.makedirs(places365_dir, exist_ok=True)

# Download weights if not already present
if not os.path.exists(weight_path):
    print(f"Downloading model weights to {weight_path}...")
    !wget -q -O {weight_path} {weights_url}
    print("Download complete.")
else:
    print(f"Model weights already exist at {weight_path}.")

# Download categories if not already present
if not os.path.exists(category_path):
    print(f"Downloading categories file to {category_path}...")
    !wget -q -O {category_path} {category_url}
    print("Download complete.")
else:
    print(f"Categories file already exist at {category_path}.")

### Load Places365 ResNet50 Model

This section contains the function to load the pre-trained ResNet50 model and its associated categories.

In [ ]:
def load_places365_resnet50(weight_path, category_path):
    model = tv_models.resnet50(num_classes=365)

    # Check if the weight file exists before loading
    if not os.path.exists(weight_path):
        raise FileNotFoundError(f"Model weights not found at: {weight_path}. Please ensure it's downloaded.")

    checkpoint = torch.load(weight_path, map_location=device)
    state_dict = checkpoint['state_dict']

    # Remove 'module.' prefix if present in the state_dict keys
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('module.'):
            new_state_dict[k[7:]] = v
        else:
            new_state_dict[k] = v

    model.load_state_dict(new_state_dict)
    model.eval()  # Set model to evaluation mode
    model.to(device)

    # Read category labels
    classes = []
    # Check if the category file exists before opening
    if not os.path.exists(category_path):
        raise FileNotFoundError(f"Category file not found at: {category_path}. Please ensure it's downloaded.")

    with open(category_path) as f:
        for line in f:
            # Original format is like: "/a/airfield 0"
            line = line.strip().split(' ')[0][3:]
            classes.append(line)

    return model, classes

model_places365, classes_places365 = load_places365_resnet50(weight_path, category_path)
print("Places365 ResNet50 Model loaded. Number of classes:", len(classes_places365))

In [ ]:
# 安装 Google Generative AI 客户端库
!pip install -q -U google-generativeai

import google.generativeai as genai
from google.colab import userdata

# 从 Colab secrets 获取 API 密钥
# 请确保你已经在 Colab Secrets 面板中添加了一个名为 'GOOGLE_API_KEY' 的秘密，并将你的 API 密钥粘贴到其中。

# Check if GOOGLE_API_KEY is already defined in the global scope.
# If not, attempt to fetch it from userdata.
# This prevents re-fetching and potential timeout if it's already available.
if 'GOOGLE_API_KEY' not in globals():
    try:
        GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
        print("GOOGLE_API_KEY successfully fetched from Colab secrets.")
    except Exception as e:
        print(f"Error fetching GOOGLE_API_KEY from Colab secrets: {e}")
        print("Please ensure your API key is correctly configured in the Colab Secrets panel and you are running in the Colab UI.")

# Configure the genai library if GOOGLE_API_KEY is available.
if 'GOOGLE_API_KEY' in globals() and GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    print("Google Generative AI configured.")
else:
    print("Warning: GOOGLE_API_KEY is not available. Please set your API key.")

### 2. 初始化 Gemini Pro Vision 模型型

我们将使用 `gemini-pro-vision` 模型，它是为处理多模态输入（图像和文本）而设计的。

In [ ]:
# 挂载 Google Drive 以访问文件
from google.colab import drive
drive.mount('/content/drive')

定义文件路径

In [ ]:
import os
import requests
from PIL import Image
from io import BytesIO

# 定义输入和输出文件夹路径
source_folder = '/content/drive/MyDrive/CES分析/图片/图片下载-圆明园/西部景区-月地云居'
output_folder = '/content/drive/MyDrive/CES分析/图片分析测试'

# 确保输出文件夹存在，如果不存在则创建
os.makedirs(output_folder, exist_ok=True)

print(f"源图片文件夹: {source_folder}")
print(f"结果输出文件夹: {output_folder}")

找到所有图片

In [ ]:
image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff', '.webp')
image_files = []

for filename in os.listdir(source_folder):
    if filename.lower().endswith(image_extensions):
        image_files.append(os.path.join(source_folder, filename))

if not image_files:
    print("在指定文件夹中没有找到任何图片文件。")
else:
    print(f"找到 {len(image_files)} 张图片进行分析。")
    for img_path in image_files:
        print(f"  - {os.path.basename(img_path)}")

In [ ]:
!pip install xlsxwriter

分析图片

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import os
from PIL import Image
from io import BytesIO
import json # Import json for structured output
import pandas as pd # Import pandas for DataFrame creation
from tenacity import retry, stop_after_attempt, wait_fixed # Import tenacity

# Ensure API key is configured (if previous cells not run, ensure here)
# GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY') # Assuming GOOGLE_API_KEY is configured via 92fdde1e
# genai.configure(api_key=GOOGLE_API_KEY) # Assuming API configuration is done via 92fdde1e

# Ensure model is initialized, using the correct available image model
model = genai.GenerativeModel('gemini-2.5-flash-image') # Keeping the user's explicitly defined model for now.

# NEW PROMPT: Requesting JSON output for structured analysis
prompt = """
作为专业景观分析师，分析图片并严格按以下JSON格式输出，仅返回JSON字符串，无其他文字。

1.  **场景有效性判断**: 判断图片是否为自然风光或含自然元素的户外场景。
    *   **有效场景**: "有效场景"。
    *   **无效场景**: "无效场景"，并提供简要 `理由` (若为截图、UI、图表、纯文本、室内或非自然风光)。

2.  **详细分析 (若为有效场景)**:
    *   **图片分类**: 高层次分类 (如“自然山水”、“城市公园”、“乡村田野”，或“其他户外场景”)。
    *   **主要景观元素**: 识别主要元素 (如山、水、树木、建筑、天空、道路、动物、人物)，列表。如无，填 []。
    *   **各项特征评估 (1-5分，1最低5最高) 及理由**:
        *   **自然度**: 自然程度；人造痕迹。
        *   **开阔度**: 视野开阔度；广阔天空/水面/远景。
        *   **复杂性**: 元素数量/种类；构图复杂度。
        *   **碎片化**: 元素分散程度；整体感。
        *   **安全性**: 传达的安全/平静/危险感。
        *   **偏好度**: 喜爱程度及理由。

**输出格式示例 (有效场景):**
```json
{
  "场景有效性判断": "有效场景",
  "图片分类": "城市公园景观",
  "主要景观元素": ["水面", "树木", "建筑", "人物", "天空"],
  "自然度": {
    "等级": 4,
    "理由": "水面和树木占比大，虽然有建筑和人物，但自然元素占据主导。"
  },
  "开阔度": {
    "等级": 3,
    "理由": "水面提供了开阔感，但被近处的树木和远处的建筑限制了部分视野。"
  },
  "复杂性": {
    "等级": 3,
    "理由": "元素种类较丰富，包括自然和人造元素，构图较为均衡。"
  },
  "碎片化": {
    "等级": 2,
    "理由": "画面整体感强，没有过多分散的小元素。"
  },
  "安全性": {
    "等级": 5,
    "理由": "环境整洁，光线明亮，没有潜在危险，给人安全舒适的感觉。"
  },
  "偏好度": {
    "等级": 4,
    "理由": "景色优美，有水有树，环境宜人，是人们乐于休闲的场所。"
  }
}
```

**输出格式示例 (无效场景):**
```json
{
  "场景有效性判断": "无效场景",
  "理由": "这张图片是一个手机界面截图，不属于自然风光。"
}
```
"""

all_analyses = [] # This list will store dictionaries of parsed analysis results

# Define retry settings for the generate_content call
# It will retry up to 3 times, waiting 10 seconds between attempts
@retry(stop=stop_after_attempt(3), wait=wait_fixed(10))
def generate_content_with_retry(model_instance, prompt_parts):
    return model_instance.generate_content(prompt_parts)

# Loop to process images
for image_path in image_files:
    try:
        with open(image_path, 'rb') as f:
            img_data = f.read()
        image = Image.open(BytesIO(img_data))

        # --- NEW: Resize image for faster processing and to avoid timeouts ---
        max_size = (1024, 1024) # Define a maximum size for images
        image.thumbnail(max_size, Image.Resampling.LANCZOS) # Resize while maintaining aspect ratio
        # --- END NEW ---

        print(f"\n正在分析图片: {os.path.basename(image_path)}")

        # Use the retry mechanism for content generation
        response = generate_content_with_retry(model, [prompt, image])

        # --- FIX: Remove Markdown fences before JSON parsing ---
        analysis_text = response.text.strip()
        if analysis_text.startswith('```json') and analysis_text.endswith('```'):
            analysis_text = analysis_text[len('```json'):-len('```')].strip()
        # --- END FIX ---

        # Attempt to parse the JSON output
        try:
            parsed_analysis = json.loads(analysis_text)
            current_image_analysis = {"图片ID/名称": os.path.basename(image_path)}

            # Flatten the JSON structure into a single dictionary for easier DataFrame creation
            current_image_analysis["场景有效性判断"] = parsed_analysis.get("场景有效性判断", "未识别")

            if current_image_analysis["场景有效性判断"] == "有效场景":
                current_image_analysis["图片分类"] = parsed_analysis.get("图片分类", "未识别")
                current_image_analysis["主要景观元素"] = ", ".join(parsed_analysis.get("主要景观元素", [])) # Join list for single cell
                for feature_name in ["自然度", "开阔度", "复杂性", "碎片化", "安全性", "偏好度"]:
                    feature_data = parsed_analysis.get(feature_name, {})
                    current_image_analysis[f"{feature_name}打分"] = feature_data.get("等级", "未识别")
                    current_image_analysis[f"{feature_name}理由"] = feature_data.get("理由", "未识别或格式不匹配")
                current_image_analysis["无效理由"] = "" # No invalid reason for valid scene
            else: # 无效场景
                current_image_analysis["图片分类"] = ""
                current_image_analysis["主要景观元素"] = ""
                for feature_name in ["自然度", "开阔度", "复杂性", "碎片化", "安全性", "偏好度"]:
                    current_image_analysis[f"{feature_name}打分"] = ""
                    current_image_analysis[f"{feature_name}理由"] = ""
                current_image_analysis["无效理由"] = parsed_analysis.get("理由", "未识别或格式不匹配")

            all_analyses.append(current_image_analysis)

            # Save the raw JSON output to a file for verification
            base_filename = os.path.splitext(os.path.basename(image_path))[0]
            output_filepath = os.path.join(output_folder, f'{base_filename}_analysis.json') # Save as JSON
            with open(output_filepath, 'w', encoding='utf-8') as outfile:
                json.dump(parsed_analysis, outfile, ensure_ascii=False, indent=2)
            print(f"分析结果已保存到: {output_filepath}")

        except json.JSONDecodeError as json_err:
            print(f"解析图片 {os.path.basename(image_path)} 的JSON结果时发生错误: {json_err}")
            print(f"原始输出: {analysis_text[:500]}...") # Print partial raw output for debugging
            # If JSON parsing fails, save the raw text and mark as error
            base_filename = os.path.splitext(os.path.basename(image_path))[0]
            output_filepath = os.path.join(output_folder, f'{base_filename}_analysis_error.txt')
            with open(output_filepath, 'w', encoding='utf-8') as outfile:
                outfile.write(analysis_text)
            all_analyses.append({
                "图片ID/名称": os.path.basename(image_path),
                "场景有效性判断": "解析错误",
                "无效理由": f"JSON解析失败: {json_err}",
                # Fill other fields as empty or 'N/A'
                "图片分类": "",
                "主要景观元素": "",
                "自然度打分": "", "自然度理由": "",
                "开阔度打分": "", "开阔度理由": "",
                "复杂性打分": "", "复杂性理由": "",
                "碎片化打分": "", "碎片化理由": "",
                "安全性打分": "", "安全性理由": "",
                "偏好度打分": "", "偏好度理由": ""
            })


    except Exception as e:
        print(f"分析图片 {os.path.basename(image_path)} 时发生错误: {e}")
        all_analyses.append({
            "图片ID/名称": os.path.basename(image_path),
            "场景有效性判断": "处理错误",
            "无效理由": f"处理异常: {e}",
            # Fill other fields as empty or 'N/A'
            "图片分类": "",
            "主要景观元素": "",
            "自然度打分": "", "自然度理由": "",
            "开阔度打分": "", "开阔度理由": "",
            "复杂性打分": "", "复杂性理由": "",
            "碎片化打分": "", "碎片化理由": "",
            "安全性打分": "", "安全性理由": "",
            "偏好度打分": "", "偏好度理由": ""
        })

print("\n所有图片分析完成！")

# Convert the collected analyses into a DataFrame
df_analyses = pd.DataFrame(all_analyses)

In [ ]:
!pip install tenacity

In [ ]:
import pandas as pd
import os
from datetime import datetime

# Recalculate summary data from all_analyses
total_images_count = len(all_analyses)
valid_images_count = sum(1 for a in all_analyses if a['场景有效性判断'] == '有效场景')
invalid_images_count = total_images_count - valid_images_count

summary_data = {
    '统计项': ['总图片数量', '有效（室外）图片数量', '排除（室内或无效）图片数量'],
    '数量': [total_images_count, valid_images_count, invalid_images_count]
}
df_summary = pd.DataFrame(summary_data)

# Generate a timestamp for the filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
excel_filename = f'image_analysis_results_{timestamp}.xlsx'
excel_output_path = os.path.join(output_folder, excel_filename)

# Create an Excel writer object
with pd.ExcelWriter(excel_output_path, engine='xlsxwriter') as writer:
    # Write the main analysis DataFrame to a sheet named 'Detailed Analysis'
    df_analyses.to_excel(writer, sheet_name='Detailed Analysis', index=False)

    # Write the summary DataFrame to a sheet named 'Summary'
    df_summary.to_excel(writer, sheet_name='Summary', index=False)

print(f"所有分析结果已成功导出到 Excel 文件: {excel_output_path}")

# Display the summary DataFrame as well
display(df_summary)


In [ ]:
from torchvision import transforms
from PIL import Image # Ensure PIL is imported for Image.open

### Places365 Image Preprocessing and Prediction Function

This section defines the image transformations required for the Places365 ResNet50 model and a function to predict the top-k categories for a given image.

In [ ]:
# Places365 common preprocessing
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def predict_image(image_path, model, classes, topk=5):
    try:
        img = Image.open(image_path).convert("RGB")
    except Exception as e:
        return {"image_path": image_path, "error": str(e)}

    input_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.nn.functional.softmax(logits, dim=1)
        top_probs, top_idxs = torch.topk(probs, k=topk, dim=1)

    top_probs = top_probs.cpu().numpy()[0]
    top_idxs = top_idxs.cpu().numpy()[0]

    result = {
        "image_path": image_path,
        "image_name": os.path.basename(image_path)
    }

    for i, (idx, prob) in enumerate(zip(top_idxs, top_probs), start=1):
        result[f"top{i}_label"] = classes[idx]
        result[f"top{i}_prob"] = float(prob)

    return result

places365_predictions = []

print("Starting Places365 predictions...")
for img_file in image_files:
    prediction_result = predict_image(img_file, model_places365, classes_places365)
    places365_predictions.append(prediction_result)
    print(f"Predicted for: {os.path.basename(img_file)}")

print("Places365 predictions complete!")

df_places365_predictions = pd.DataFrame(places365_predictions)
display(df_places365_predictions.head())

汇总生成excel

### 4. 解释和注意事项

*   **Gemini Pro Vision 的优势**：与 Vision API 侧重于对象检测和标签识别不同，MLLM 能够更好地理解图像的**上下文**和**语义**，并根据自然语言指令进行**推理**。因此，它能够直接对“自然度”、“开阔度”等更抽象的概念进行判断，并给出解释。
*   **提示工程 (Prompt Engineering)**：您提供给模型的提示（`prompt`）非常重要。通过精心设计的提示，您可以引导模型关注特定的方面，并以您希望的格式返回结果。
*   **主观性**：尽管 MLLM 可以进行这些判断，但“安全性”和“偏好度”等仍然带有一定的主观性，模型的判断是基于其训练数据和学习到的模式。对于高度专业或精确的心理学、美学分析，可能仍需要结合人类评估或专门训练的模型。

运行上述代码后，您将看到 Gemini 模型对图片景观元素的识别以及对各项场景特征的综合判断和解释。